# Create Buses for Routes
Objective: Create the desired number of buses accoarding to routes, depots and types \
Author: Sven Vorgheim \
Disclaimer: The creation of this file was aided by ChatGPT

For each route in merged_routes.csv create the appropriate number of buses of type Type
Make them start at TZA Depot
Deadhead to first station of route
Perform route nr_of_repetitions-nr_of_buses times
deadhead to depot tza
Stagger each bus by period


In [ ]:
# AI generated on 2026-06-28

import math
import pandas as pd
import xml.etree.ElementTree as ET

CSV_FILE = "merged_routes.csv"
ROUTES_FILE = "cicero_mueller_routes.rou.xml"
OUTPUT_FILE = "vehicles.rou.xml"

# --------------------------------------------------------------------
# Change these to your actual deadhead route IDs
# --------------------------------------------------------------------

DEPOT_ROUTE_TO = "tza_to_{route}"
DEPOT_ROUTE_FROM = "{route}_to_tza"

# --------------------------------------------------------------------

tree = ET.parse(ROUTES_FILE)
root = tree.getroot()

route_lookup = {}

for route in root.findall("route"):
    route_lookup[route.attrib["id"]] = route


routes_root = ET.Element(
    "routes",
    {
        "xmlns:xsi": "http://www.w3.org/2001/XMLSchema-instance",
        "xsi:noNamespaceSchemaLocation":
            "http://sumo.dlr.de/xsd/routes_file.xsd"
    }
)

# Copy vTypes
for vtype in root.findall("vType"):
    routes_root.append(vtype)

df = pd.read_csv(CSV_FILE)

for _, row in df.iterrows():

    route_id = str(row["route"])

    if route_id not in route_lookup:
        print(f"Missing route {route_id}")
        continue

    n_buses = int(row["nr_of_buses"])
    total_reps = int(row["nr_of_repetitions"])

    flow_begin = float(row["flow_begin"])
    period = float(row["period"])

    vehicle_type = str(row["Type"])

    base = total_reps // n_buses
    extra = total_reps % n_buses

    for i in range(n_buses):

        repetitions = base
        if i < extra:
            repetitions += 1

        depart = flow_begin + i * period

        vehicle = ET.SubElement(
            routes_root,
            "vehicle",
            {
                "id": f"{route_id}_{i}",
                "type": vehicle_type,
                "depart": str(depart)
            }
        )

        # ------------------------------------------------------------
        # Deadhead from depot to first stop
        # ------------------------------------------------------------

        ET.SubElement(
            vehicle,
            "route",
            {
                "refId": DEPOT_ROUTE_TO.format(route=route_id)
            }
        )

        # ------------------------------------------------------------
        # Perform service route repetitions
        # ------------------------------------------------------------

        for _ in range(repetitions):

            ET.SubElement(
                vehicle,
                "route",
                {
                    "refId": route_id
                }
            )

        # ------------------------------------------------------------
        # Deadhead back to depot
        # ------------------------------------------------------------

        ET.SubElement(
            vehicle,
            "route",
            {
                "refId": DEPOT_ROUTE_FROM.format(route=route_id)
            }
        )


ET.indent(routes_root, space="    ")

ET.ElementTree(routes_root).write(
    OUTPUT_FILE,
    encoding="utf-8",
    xml_declaration=True
)

print(f"Wrote {OUTPUT_FILE}")